[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/how-to-train-your-models/distributed-jaxlings/blob/main/chapters/chapter_02_data_parallelism.ipynb)

# Chapter 2: Data Parallelism

> **Course: Distributed Training — From Concepts to JAX**

---

## Learning Objectives

- Explain the data parallel training loop
- Express gradient computations using Noam/einsum notation
- Describe the Ring-AllReduce algorithm and its bandwidth efficiency
- Distinguish synchronous vs asynchronous gradient updates
- Implement gradient accumulation to simulate larger batch sizes
- Explain ZeRO stages and the memory savings they provide


---
## Setup


In [13]:
import jax
import jax.numpy as jnp
import numpy as np
import optax

from functools import partial
from jaxtyping import Array, Float
from judge import Judge


judge = Judge("Chapter 2")
print(f"JAX devices: {jax.devices()}")
print("Judge ready!")

JAX devices: [CpuDevice(id=0)]
Judge ready!


---
## 1. The Data Parallel Idea

Data parallelism is the simplest and most widely used form of distributed training.

**Core idea:**
1. Each worker (GPU) holds a **full copy** of the model
2. The global mini-batch is **split** across workers — each sees a different micro-batch
3. Each worker computes a forward + backward pass independently
4. Gradients are **averaged** across all workers (AllReduce)
5. Each worker applies the same gradient update → models stay in sync

```
Global batch = [b0, b1, b2, b3]  (4 micro-batches)

GPU 0: model_copy | b0 → grad_0 ─┐
GPU 1: model_copy | b1 → grad_1 ─┤→ AllReduce → avg_grad → update all
GPU 2: model_copy | b2 → grad_2 ─┤
GPU 3: model_copy | b3 → grad_3 ─┘
```

**Effective batch size** = micro_batch_size × n_workers.

### Gradients

For a linear layer `Y_BxSxF = X_BxSxD @ W_DxF`, the gradient of the loss w.r.t. `W`
sums each input activation `X[b,s,d]` scaled by the upstream error signal `δ[b,s,f]`,
across every sample and token in the batch:

```
dL/dW[d,f]  =  Σ over all (b, s):  X[b,s,d] × δ[b,s,f]
  dW_DxF   =  einsum("BSD, BSF -> DF", X_BxSxD, δ_BxSxF)
```

Intuitively: how much did input feature `d` contribute to output feature `f`, averaged
over the whole batch? That's your gradient.

In data parallelism, each worker computes this sum over its own micro-batch.
**AllReduce then averages these local gradients** — that is the only communication step.

**Axis legend:** `B`=batch · `S`=sequence · `D`=model dim · `F`=feed-forward dim · `W`=workers · `P`=params


---
### Exercise 1: Data Parallel Gradient Averaging

Each worker computes `dL/dW` over its micro-batch.
Implement `worker_grad_W` and `data_parallel_average`.

```
TODO: implement both functions below
```


In [14]:
def worker_grad_W(X_BxSxD: Float[Array, "B S D"], delta_BxSxF: Float[Array, "B S F"]) -> Float[Array, "D F"]:
    """
    Compute the gradient of a linear layer's weight matrix on one micro-batch.

    dL/dW = einsum("b s d, b s f -> d f", X_BxSxD, delta_BxSxF)

    Args:
        X_BxSxD:     activations,       shape [batch, seq, d_model]
        delta_BxSxF: upstream gradient, shape [batch, seq, d_ff]

    Returns:
        dW_DxF of shape [d_model, d_ff]
    """
    # TODO: return jnp.einsum("bsd,bsf->df", X_BxSxD, delta_BxSxF)
    return jnp.einsum("bsd,bsf->df", X_BxSxD, delta_BxSxF)


def data_parallel_average(worker_grads: jnp.ndarray) -> jnp.ndarray:
    """
    Average gradients from all workers (simulates AllReduce).

    Args:
        worker_grads: stacked gradients, shape [n_workers, d_model, d_ff]
                      axis 0 = w (workers)

    Returns:
        averaged gradient, shape [d_model, d_ff]
    """
    # TODO: return jnp.mean(worker_grads, axis=0)
    #       equivalently: einsum("w d f -> d f", worker_grads) / n_workers
    return jnp.mean(worker_grads, axis=0)


# ── Test ─────────────────────────────────────────────────────────────────────
key = jax.random.PRNGKey(0)
B, S, D, F, W = 4, 6, 8, 16, 3   # batch, seq, d_model, d_ff, n_workers

# Each worker has its own micro-batch X and upstream gradient delta
Xs     = jax.random.normal(key, (W, B, S, D))
deltas = jax.random.normal(key, (W, B, S, F))

local_grads = jnp.stack([worker_grad_W(Xs[w], deltas[w]) for w in range(W)])  # [W, D, F]
avg_grad    = data_parallel_average(local_grads)                                # [D, F]

expected_avg = jnp.mean(
    jnp.stack([jnp.einsum("bsd,bsf->df", Xs[w], deltas[w]) for w in range(W)]), axis=0
)

judge.check("Ex1a: worker_grad_W shape",         local_grads.shape, (W, D, F))
judge.check("Ex1b: data_parallel_average shape", avg_grad.shape,    (D, F))
judge.check("Ex1c: averaged values correct",     avg_grad,          expected_avg)

✅ Ex1a: worker_grad_W shape: PASSED
✅ Ex1b: data_parallel_average shape: PASSED
✅ Ex1c: averaged values correct: PASSED


True

<details>
<summary>💡 Hint</summary>

```python
# worker_grad_W
return jnp.einsum("bsd,bsf->df", X, delta)

# data_parallel_average
return jnp.mean(worker_grads, axis=0)
```
</details>


---
## 2. AllReduce: Synchronizing Gradients

AllReduce **reduces tensors across all workers and distributes the result back to all workers**.

### Naive AllReduce (parameter server)
```
All workers → send grad to server → server sums → broadcast back
```
- Server bandwidth: O(N × model_size) — bottleneck!

### Ring-AllReduce
Workers are arranged in a ring. Two phases:

**Phase 1 — Reduce-Scatter** (N-1 steps):
Each worker sends chunk `(rank - step) % N` rightward and **adds** the received chunk.
After N-1 steps, each worker owns the **fully reduced** version of one chunk.

**Phase 2 — AllGather** (N-1 steps):
Each worker sends its complete chunk rightward, **overwriting** the receiver's copy.
After N-1 steps, every worker has all chunks → full result.

**Bandwidth efficiency:** each worker sends/receives exactly `2(N-1)/N × S` bytes.
As N → ∞, this converges to **2S** — independent of N. Ring-AllReduce is **bandwidth-optimal**.

```
Ring with 4 workers, gradient split into 4 chunks [A, B, C, D]

Initial:  W0=[A,B,C,D]  W1=[A,B,C,D]  W2=[A,B,C,D]  W3=[A,B,C,D]

--- Reduce-Scatter (3 steps) ---
Each worker ends up with one fully-reduced chunk.

--- AllGather (3 steps) ---
Each worker broadcasts its chunk around the ring.

Final: all workers hold [ΣA, ΣB, ΣC, ΣD]
```


---
### Exercise 2: Ring-AllReduce

Implement the two-phase Ring-AllReduce. Workers hold flat gradient vectors
(think of them as flattened `W[d,f]` from Exercise 1).

```
TODO: fill in the TODO lines inside ring_allreduce
```


In [15]:
def ring_allreduce(worker_data: list) -> list:
    """
    Ring-AllReduce over a list of 1-D JAX arrays (one per worker).

    Args:
        worker_data: list of jnp.ndarray, each shape [p]  (p = n_params)

    Returns:
        list of jnp.ndarray — each worker holds the element-wise SUM.
        (caller divides by n_workers to get the mean)
    """
    n = len(worker_data)
    p = worker_data[0].shape[0]
    assert p % n == 0, "p must be divisible by n for this simplified version"
    chunk = p // n

    # chunks[w][i] = slice i of worker w's gradient vector
    chunks = [
        [worker_data[w][i * chunk:(i + 1) * chunk] for i in range(n)]
        for w in range(n)
    ]

    # ── Phase 1: Reduce-Scatter ──────────────────────────────────────────────
    for step in range(n - 1):
        new_chunks = [row[:] for row in chunks]
        for w in range(n):
            src   = (w - 1) % n                # left neighbour
            # TODO: which chunk index does 'src' send to us in this step?
            idx   = 0  # TODO: (w - 1 - step) % n
            # TODO: accumulate: new_chunks[w][idx] = chunks[w][idx] + chunks[src][idx]
        chunks = new_chunks

    # ── Phase 2: AllGather ───────────────────────────────────────────────────
    for step in range(n - 1):
        new_chunks = [row[:] for row in chunks]
        for w in range(n):
            src   = (w - 1) % n
            # TODO: which chunk index does 'src' send to us in this step?
            idx   = 0  # TODO: (w - step) % n
            # TODO: overwrite: new_chunks[w][idx] = chunks[src][idx]
        chunks = new_chunks

    return [jnp.concatenate(chunks[w]) for w in range(n)]


# ── Test ─────────────────────────────────────────────────────────────────────
key = jax.random.PRNGKey(1)
W, P = 4, 8   # n_workers, n_params (P divisible by W)
data = [jax.random.normal(key, (P,)) for _ in range(W)]

true_sum = jnp.sum(jnp.stack(data), axis=0)   # shape [P]
results  = ring_allreduce(data)

judge.check("Ex2a: worker 0 holds the sum",   results[0], true_sum)
judge.check("Ex2b: all workers agree",
            jnp.allclose(results[0], results[1]) and jnp.allclose(results[1], results[3]),
            True)

❌ Ex2a: worker 0 holds the sum: FAILED
   got:      Array([-0.15443718,  0.08470728, -0.13598049, -0.15503626,  1.2666674 ,
        0.14829758,  2.1415603 ,  1.0026742 ], dtype=float32)
   expected: Array([-0.61774874,  0.3388291 , -0.54392195, -0.620145  ,  5.0666695 ,
        0.5931903 ,  8.566241  ,  4.010697  ], dtype=float32)
✅ Ex2b: all workers agree: PASSED


True

<details>
<summary>💡 Hint — Reduce-Scatter</summary>

```python
idx = (w - 1 - step) % n
new_chunks[w][idx] = chunks[w][idx] + chunks[src][idx]
```
</details>

<details>
<summary>💡 Hint — AllGather</summary>

```python
idx = (w - step) % n
new_chunks[w][idx] = chunks[src][idx]
```
</details>


---
## 3. Synchronous vs Asynchronous Training

| | Synchronous (SSP) | Asynchronous (ASP) |
|---|---|---|
| **Gradient update** | All workers sync before step | Workers update independently |
| **Convergence** | Identical to single-GPU math | Stale gradients → noise |
| **Stragglers** | Slowest worker blocks all | No blocking |
| **Used by** | JAX `pmap`, PyTorch DDP | Hogwild!, some RL systems |

Modern LLM training is **always synchronous** — stale gradients hurt convergence too much at scale.


---
## 4. Gradient Accumulation

When GPU memory limits the per-step batch size, accumulate gradients over several
micro-batches before updating:

```
effective_batch = micro_batch × accumulation_steps × n_workers
```

In JAX functional style this means summing `jax.grad` outputs before passing
them to `optax`:

```python
# pseudo-code
grads_acc = zero_grads(params)
for micro_batch in micro_batches:
    grads_acc = add_grads(grads_acc, jax.grad(loss_fn)(params, micro_batch))
grads_avg = scale_grads(grads_acc, 1 / accumulation_steps)
updates, opt_state = optimizer.update(grads_avg, opt_state)
params = optax.apply_updates(params, updates)
```

### Gradient accumulation with einsum

For a linear layer, the accumulated gradient is:

```
dL/dW  +=  einsum("b s d, b s f -> d f",  X_micro,  δ_micro)
```

Summing this over `accumulation_steps` micro-batches — then dividing by the total —
is mathematically identical to computing the gradient on the full effective batch.


---
### Exercise 3: Gradient Accumulation in JAX

Train a linear model `Y[b,s,f] = X[b,s,d] @ W[d,f]` with gradient accumulation.
Use `jax.grad` for differentiation and `optax.adam` for the optimizer.

```
TODO: implement accumulate_and_step
```


In [16]:
def mse_loss(W: jnp.ndarray, X: jnp.ndarray, Y: jnp.ndarray) -> jnp.ndarray:
    """MSE loss for a linear layer.  Axes: b=batch, s=seq, d=in, f=out."""
    Y_hat = jnp.einsum("bsd,df->bsf", X, W)
    return jnp.mean((Y_hat - Y) ** 2)


def accumulate_and_step(
    W: jnp.ndarray,
    opt_state,
    optimizer,
    micro_Xs: list,
    micro_Ys: list,
) -> tuple:
    """
    Accumulate gradients over micro-batches, then apply one optimizer step.

    Args:
        W:         weight matrix [d_model, d_ff]
        opt_state: current optax optimizer state
        optimizer: optax optimizer (e.g. optax.adam(...))
        micro_Xs:  list of micro-batch inputs,  each [b, s, d_model]
        micro_Ys:  list of micro-batch targets, each [b, s, d_ff]

    Returns:
        (new_W, new_opt_state, step_loss)

    Steps:
        1. For each micro-batch, compute grad = jax.grad(mse_loss)(W, X, Y)
        2. Accumulate (sum) the grads
        3. Divide accumulated grad by len(micro_Xs)  ← scale to mean
        4. Call optimizer.update(avg_grad, opt_state) → updates, new_opt_state
        5. new_W = optax.apply_updates(W, updates)
    """
    n_acc = len(micro_Xs)

    # TODO: initialise accumulated gradient to zeros with same shape as W
    grad_acc = None  # TODO: jnp.zeros_like(W)

    step_loss = 0.0
    for X_mb, Y_mb in zip(micro_Xs, micro_Ys):
        # TODO: compute loss and gradient for this micro-batch
        loss, g = None, None  # TODO: jax.value_and_grad(mse_loss)(W, X_mb, Y_mb)
        step_loss += loss
        # TODO: accumulate gradient
        # grad_acc = grad_acc + g

    # TODO: scale to average
    avg_grad = None  # TODO: grad_acc / n_acc

    # TODO: optimizer step
    updates, new_opt_state = None, None  # TODO: optimizer.update(avg_grad, opt_state)
    new_W = None                         # TODO: optax.apply_updates(W, updates)

    return new_W, new_opt_state, step_loss / n_acc


# ── Test ─────────────────────────────────────────────────────────────────────
key = jax.random.PRNGKey(2)
B, S, D, F = 8, 4, 6, 3   # b, s, d_model, d_ff

W_true = jax.random.normal(key, (D, F))          # target weights
W_init = jax.random.normal(jax.random.PRNGKey(99), (D, F)) * 0.01

optimizer = optax.adam(1e-2)
opt_state = optimizer.init(W_init)
W_cur     = W_init

n_steps, n_acc = 300, 4
losses = []
for _ in range(n_steps):
    Xs = [jax.random.normal(key, (B, S, D)) for _ in range(n_acc)]
    Ys = [jnp.einsum("bsd,df->bsf", X, W_true) for X in Xs]
    W_cur, opt_state, l = accumulate_and_step(W_cur, opt_state, optimizer, Xs, Ys)
    losses.append(float(l))

judge.check("Ex3a: loss decreased",          losses[-1] < losses[0],                     True)
judge.check("Ex3b: weights converged",       jnp.allclose(W_cur, W_true, atol=0.05),     True)

TypeError: unsupported operand type(s) for +=: 'float' and 'NoneType'

<details>
<summary>💡 Hint</summary>

```python
grad_acc = jnp.zeros_like(W)
for X_mb, Y_mb in zip(micro_Xs, micro_Ys):
    loss, g = jax.value_and_grad(mse_loss)(W, X_mb, Y_mb)
    step_loss += loss
    grad_acc = grad_acc + g
avg_grad = grad_acc / n_acc
updates, new_opt_state = optimizer.update(avg_grad, opt_state)
new_W = optax.apply_updates(W, updates)
```
</details>


---
## 5. ZeRO: Zero Redundancy Optimizer

Data parallelism stores a **full model copy on every GPU** — wasteful!
ZeRO (Rajbhandari et al., 2020) eliminates this redundancy progressively:

```
              Memory per GPU (175B model, 64 GPUs)
Base DDP  ──────────────────────────────────────── 2800 GB  (full copy on every GPU)
ZeRO-1    Shard optimizer states          ──────── 730 GB   (4× reduction)
ZeRO-2    Shard optimizer states+gradients ──────  365 GB   (8× reduction)
ZeRO-3    Shard everything (weights too)  ───────   43 GB   (64× = N GPUs)
```

**ZeRO-1:** Each rank owns 1/N of the optimizer states. After AllReduce of gradients,
each rank updates its shard, then AllGather parameters.

**ZeRO-2:** Additionally shard gradients. After Reduce-Scatter, each rank owns
only the gradient shard it needs.

**ZeRO-3:** Shard weights too. Parameters are gathered on-demand during forward/backward.

**Trade-off:** extra AllGather communication, but memory savings usually dominate.

### Sharding in Noam notation

Flattened params have axis `p`. With N workers:

```
Full params:   W[p]             lives on all workers (base DDP)
ZeRO-1 shard: W[p // N]        each worker owns a contiguous slice

After local Adam update on shard s:
  W_shard[s, p//N] updated independently

AllGather reassembles:
  einsum("w q -> w q", shards)  → reshape to W[p]   (w × q = p)
```


---
### Exercise 4: ZeRO-1 Optimizer State Sharding

Each rank updates only its slice of the parameter vector using Adam,
then the full parameter vector is reconstructed via AllGather.

```
TODO: implement zero1_assign_shards and zero1_adam_step_shard
```


In [ ]:
from typing import List, Tuple


def zero1_assign_shards(n_params: int, n_ranks: int) -> List[jnp.ndarray]:
    """
    Assign parameter indices [0..n_params) to ranks as evenly as possible.

    Returns:
        list of length n_ranks, each element is a jnp.ndarray of indices
        (axis label: `p_shard` — the shard of the `p` axis owned by this rank)
    """
    # TODO: np.array_split(np.arange(n_params), n_ranks), convert each to jnp
    pass


def zero1_adam_step_shard(
    params:        jnp.ndarray,   # [p]  — full params (read-only except at shard)
    grads:         jnp.ndarray,   # [p]  — averaged gradients (from AllReduce)
    m:             jnp.ndarray,   # [p_shard]
    v:             jnp.ndarray,   # [p_shard]
    shard_idx:     jnp.ndarray,   # [p_shard]  indices into the p axis
    lr:            float,
    beta1:         float = 0.9,
    beta2:         float = 0.999,
    eps:           float = 1e-8,
    t:             int   = 1,
) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """
    Apply an Adam update to the shard of params this rank owns.

    Returns:
        (updated_params [p], new_m [p_shard], new_v [p_shard])
    """
    # TODO: g = grads[shard_idx]                              # [p_shard]
    # TODO: new_m = beta1 * m + (1 - beta1) * g
    # TODO: new_v = beta2 * v + (1 - beta2) * g ** 2
    # TODO: m_hat = new_m / (1 - beta1 ** t)
    # TODO: v_hat = new_v / (1 - beta2 ** t)
    # TODO: params = params.at[shard_idx].add(-lr * m_hat / (jnp.sqrt(v_hat) + eps))
    # TODO: return params, new_m, new_v
    pass


# ── Test ─────────────────────────────────────────────────────────────────────
P, N = 12, 4   # n_params, n_ranks
shards = zero1_assign_shards(P, N)

all_idx = jnp.sort(jnp.concatenate(shards))
judge.check("Ex4a: all params assigned",          all_idx, jnp.arange(P))
judge.check("Ex4b: shard size difference <= 1",
            int(max(len(s) for s in shards) - min(len(s) for s in shards)) <= 1, True)

params = jnp.ones(P)
grads  = jnp.full(P, 0.1)
s0     = shards[0]
m0, v0 = jnp.zeros(len(s0)), jnp.zeros(len(s0))

params, m0, v0 = zero1_adam_step_shard(params, grads, m0, v0, s0, lr=0.01, t=1)

judge.check("Ex4c: shard 0 was updated",       bool(jnp.any(params[s0] != 1.0)),         True)
judge.check("Ex4d: shard 1 untouched",         bool(jnp.all(params[shards[1]] == 1.0)),  True)

<details>
<summary>💡 Hint — zero1_assign_shards</summary>

```python
return [jnp.array(s) for s in np.array_split(np.arange(n_params), n_ranks)]
```
</details>

<details>
<summary>💡 Hint — zero1_adam_step_shard</summary>

```python
g     = grads[shard_idx]
new_m = beta1 * m + (1 - beta1) * g
new_v = beta2 * v + (1 - beta2) * g ** 2
m_hat = new_m / (1 - beta1 ** t)
v_hat = new_v / (1 - beta2 ** t)
params = params.at[shard_idx].add(-lr * m_hat / (jnp.sqrt(v_hat) + eps))
return params, new_m, new_v
```
</details>


---
## Summary


In [ ]:
judge.summary()

---
## Key Takeaways

1. **Data parallelism** replicates the model across GPUs and splits data. Each GPU
   computes `dL/dW = einsum("b s d, b s f -> d f", X, δ)` on its micro-batch;
   AllReduce averages these before the update.
2. **Ring-AllReduce** is bandwidth-optimal: each GPU's communication cost is constant
   regardless of the number of GPUs.
3. **Gradient accumulation** sums `jax.grad` outputs over micro-batches before a single
   `optimizer.update` — mathematically equivalent to training on the full effective batch.
4. **ZeRO-1/2/3** shard optimizer states, gradients, and weights respectively,
   reducing per-GPU memory by up to N× with manageable communication overhead.

---
**Next:** [Chapter 3 — Model Parallelism](./chapter_03_model_parallelism.ipynb)
